In [ ]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit

df = pd.read_csv('/final_dashboard_data.csv')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst 3 rows:")
print(df.head(3))

Shape: (1600, 10)

Columns: ['video_id', 'title', 'category_id', 'published_at', 'views', 'likes', 'comments', 'fetched_at', 'category', 'half_life_hours']

First 3 rows:
      video_id                                              title  \
0  b68HETiNO98  @SaiAbhyankkar - Pavazha Malli (Music Video) |...   
1  NHk7scrb_9I  Dhurandhar The Revenge Official Hindi Trailer ...   
2  F5W1G5Qbti0  WELCOME TO THE MOST EXPENSIVE CITY | GTA 5 GAM...   

   category_id          published_at     views   likes  comments  \
0           10  2026-03-05T13:38:57Z  10129558  189889      6370   
1           24  2026-03-07T05:31:00Z  20637519  777792     41907   
2           20  2026-03-07T05:30:41Z   6864790  645996    102483   

                   fetched_at       category  half_life_hours  
0  2026-03-07T18:47:55.962779          Music             9.30  
1  2026-03-07T18:47:55.962796  Entertainment             8.03  
2  2026-03-07T18:47:55.962800         Gaming           -20.78  


In [ ]:
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')

# Sort data properly
df['fetched_at'] = pd.to_datetime(df['fetched_at'])
df = df.sort_values(['video_id', 'fetched_at'])

# Calculate view velocity
df['views_delta'] = df.groupby('video_id')['views'].diff()
df['time_delta_hrs'] = df.groupby('video_id')['fetched_at'].diff().dt.total_seconds() / 3600
df['view_velocity'] = df['views_delta'] / df['time_delta_hrs']

# Half-life function
def exponential_decay(t, V0, lambda_decay):
    return V0 * np.exp(-lambda_decay * t)

def calculate_half_life(video_data):
    velocities = video_data['view_velocity'].dropna().values
    velocities = velocities[velocities > 0]  # remove negative values
    if len(velocities) < 3:
        return None
    t = np.arange(len(velocities))
    try:
        popt, _ = curve_fit(exponential_decay, t, velocities,
                           p0=[velocities[0], 0.1], maxfev=5000)
        V0, lambda_val = popt
        if lambda_val <= 0:
            return None
        half_life = np.log(2) / lambda_val
        return round(half_life, 2)
    except:
        return None

# Calculate half-life per video
half_lives = df.groupby('video_id').apply(calculate_half_life).reset_index()
half_lives.columns = ['video_id', 'half_life_hours']

# Remove nulls and negatives
half_lives = half_lives.dropna()
half_lives = half_lives[half_lives['half_life_hours'] > 0]

print("Sample half-lives:")
print(half_lives.head(10))
print("\nTotal videos with valid half-life:", len(half_lives))

Sample half-lives:
       video_id  half_life_hours
0   4GtRBMbpJJg             0.44
1   5RBaQlbh5Ys             0.29
2   5ZYPA93DvVU             2.69
3   7q9iq2Bf4C4             1.58
4   8RV_zhiWiws             6.09
5   8aG_TwWMtvQ             5.62
7   Aze7zTyIRtk             2.02
8   CBl2l9YAIEI             0.77
9   COiAjiu0itM             2.34
10  EUimRsgV5LA             1.42

Total videos with valid half-life: 47


In [ ]:
# Merge half-life back with video details
video_info = df.drop_duplicates('video_id')[['video_id','title','category','views','likes','comments']]
final_df = video_info.merge(half_lives, on='video_id')

# Check category summary
category_summary = final_df.groupby('category')['half_life_hours'].mean().round(2).sort_values()
print("Average Half-Life by Category (hours):")
print(category_summary)

# Save new CSV
final_df.to_csv('fixed_dashboard_data.csv', index=False)
print("\n✅ File saved as fixed_dashboard_data.csv!")
print("Total rows:", len(final_df))

Average Half-Life by Category (hours):
category
People & Blogs      1.72
Entertainment       2.62
Music               2.86
Film & Animation    3.47
Gaming              3.79
Comedy              3.88
Name: half_life_hours, dtype: float64

✅ File saved as fixed_dashboard_data.csv!
Total rows: 47


In [6]:
# Merge half-life back with ALL original columns
video_info = df.drop_duplicates('video_id')[['video_id','title','category_id','category',
                                              'published_at','views','likes','comments','fetched_at']]
final_df = video_info.merge(half_lives, on='video_id')

# Check columns
print("Columns:", final_df.columns.tolist())
print("Total rows:", len(final_df))

# Save new CSV with ALL columns
final_df.to_csv('fixed_dashboard_data.csv', index=False)
print("✅ File saved with all columns!")

Columns: ['video_id', 'title', 'category_id', 'category', 'published_at', 'views', 'likes', 'comments', 'fetched_at', 'half_life_hours']
Total rows: 47
✅ File saved with all columns!


In [7]:
print(final_df['half_life_hours'].describe())
print(final_df[['title','half_life_hours']].head(10))

count    47.000000
mean      3.020000
std       3.617434
min       0.160000
25%       0.930000
50%       2.130000
75%       4.010000
max      23.000000
Name: half_life_hours, dtype: float64
                                               title  half_life_hours
0  Saabi Bhinder - 25-26 (Official Music Video) f...             0.44
1  Sadiya Kariya (Viral Bhojpuri Song) | #Ankush ...             0.29
2  Dhurandhar The Revenge Trailer REACTION| Ranve...             2.69
3  The DARK PLANS Of The PROTOTYPE & The EX-EMPLO...             1.58
4  #video | #Khesari Lal Yadav | पत्नी खोजतानी | ...             6.09
5  Youth Trailer |  Ken Karunaas | Suraj Venjaram...             5.62
6  Aura 10/10 - Music Video | Meesaya Murukku 2 |...             2.02
7  Sheesha (  Aakhya Mai Aakh Ghali Jo Beran ) | ...             0.77
8       I Found a Secret Scary RED DOOR In MInecraft             2.34
9  😱nobita ne car tod di 🤯 indian theft auto simu...             1.42


In [8]:
final_df.to_csv('fixed_dashboard_data.csv', index=False)
print("Done!")
print(final_df.columns.tolist())


Done!
['video_id', 'title', 'category_id', 'category', 'published_at', 'views', 'likes', 'comments', 'fetched_at', 'half_life_hours']
